In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.corpus.reader import reviews
stops = set(stopwords.words('english'))

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, SimpleRNN, Dense, Dropout, Input , LSTM, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
data = pd.read_csv('/content/drive/MyDrive/Restaurant_Reviews.tsv', sep='\t')

In [ ]:
data.head()

,Review,Liked
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [ ]:
data.Review[0]

'Wow... Loved this place.'

In [ ]:

review = re.sub('[^a-zA-z]' , " " , data['Review'][0])

In [ ]:
review

'Wow    Loved this place '

In [ ]:
review = review.lower()

In [ ]:
corpus = []

for i in range(0 , 1000):
  review = re.sub('[^a-zA-z]' , " " , data['Review'][i])
  review = review.lower()
  review = review.split()
  ps = PorterStemmer()

  review = [ ps.stem(word) for word in review if not word in set(stopwords.words('english'))]

  review = ' '.join(review)

  corpus.append(review)


In [ ]:
corpus

['wow love place',
 'crust good',
 'tasti textur nasti',
 'stop late may bank holiday rick steve recommend love',
 'select menu great price',
 'get angri want damn pho',
 'honeslti tast fresh',
 'potato like rubber could tell made ahead time kept warmer',
 'fri great',
 'great touch',
 'servic prompt',
 'would go back',
 'cashier care ever say still end wayyy overpr',
 'tri cape cod ravoli chicken cranberri mmmm',
 'disgust pretti sure human hair',
 'shock sign indic cash',
 'highli recommend',
 'waitress littl slow servic',
 'place worth time let alon vega',
 'like',
 'burritto blah',
 'food amaz',
 'servic also cute',
 'could care less interior beauti',
 'perform',
 'right red velvet cake ohhh stuff good',
 'never brought salad ask',
 'hole wall great mexican street taco friendli staff',
 'took hour get food tabl restaur food luke warm sever run around like total overwhelm',
 'worst salmon sashimi',
 'also combo like burger fri beer decent deal',
 'like final blow',
 'found place acc

In [ ]:
VOCAB_SIZE   = 5000
MAX_LEN      = 200
EMBED_DIM    = 64
BATCH_SIZE   = 32
EPOCHS       = 20

In [ ]:
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(corpus)

In [ ]:
sequences = tokenizer.texts_to_sequences(corpus)
padded    = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')


In [ ]:
labels = data.iloc[:, 1].values.astype(int)

split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    padded, labels, test_size=0.2, random_state=42
)
print(f"\nTrain size : {X_train.shape}")
print(f"Test  size : {X_test.shape}")


Train size : (800, 200)
Test  size : (200, 200)


LSTM MODEL

In [ ]:
lstm_model = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),
    LSTM(64),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm_model.summary()

history_LSTM = lstm_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,

    )

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 64)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 353,089 (1.35 MB)

 Trainable params: 353,089 (1.35 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 156ms/step - accuracy: 0.5078 - loss: 0.6936 - val_accuracy: 0.4563 - val_loss: 0.6961
Epoch 2/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 162ms/step - accuracy: 0.5172 - loss: 0.6942 - val_accuracy: 0.4563 - val_loss: 0.6950
Epoch 3/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 97ms/step - accuracy: 0.5172 - loss: 0.6930 - val_accuracy: 0.4563 - val_loss: 0.6942
Epoch 4/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 96ms/step - accuracy: 0.5172 - loss: 0.6930 - val_accuracy: 0.4563 - val_loss: 0.6954
Epoch 5/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 96ms/step - accuracy: 0.5172 - loss: 0.6928 - val_accuracy: 0.4563 - val_loss: 0.6957
Epoch 6/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 96ms/step - accuracy: 0.5172 - loss: 0.6929 - val_accuracy: 0.4563 - val_loss: 0.6965
Epoch 7/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 113ms/step - accuracy: 0.5172 - loss: 0.6929 - val_accuracy: 0.4563 - val_loss: 0.6973
Epoch 8/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 162ms/step - accuracy: 0.5172 - loss: 0.6927 - val_accuracy: 0.4563

In [ ]:
model_name = 'LSTM'

y_prob = lstm_model.predict(X_test ).ravel()
y_pred = (y_prob >= 0.5).astype(int)
print(f'\n{'='*50}')
print(f'  {model_name} Results')
print(f'{'='*50}')
print(classification_report(y_test, y_pred))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')

7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step

  LSTM Results
              precision    recall  f1-score   support

           0       0.48      1.00      0.65        96
           1       0.00      0.00      0.00       104

    accuracy                           0.48       200
   macro avg       0.24      0.50      0.32       200
weighted avg       0.23      0.48      0.31       200

Confusion Matrix:
[[ 96   0]
 [104   0]]
Accuracy: 0.4800


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
BI_lstm_model = Sequential([
    Input(shape=(MAX_LEN,)),

    Embedding(input_dim=VOCAB_SIZE,
              output_dim=EMBED_DIM),

    Bidirectional(LSTM(64)),

    Dense(1, activation='sigmoid')
])

BI_lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
BI_lstm_model.summary()

history_BI = BI_lstm_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,

    )

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 200, 64)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 386,177 (1.47 MB)

 Trainable params: 386,177 (1.47 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 9s 258ms/step - accuracy: 0.5109 - loss: 0.6917 - val_accuracy: 0.4688 - val_loss: 0.6917
Epoch 2/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 183ms/step - accuracy: 0.6047 - loss: 0.6689 - val_accuracy: 0.6812 - val_loss: 0.6665
Epoch 3/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 223ms/step - accuracy: 0.8391 - loss: 0.5657 - val_accuracy: 0.6875 - val_loss: 0.5820
Epoch 4/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 223ms/step - accuracy: 0.8281 - loss: 0.4304 - val_accuracy: 0.6750 - val_loss: 0.5735
Epoch 5/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 207ms/step - accuracy: 0.9219 - loss: 0.3105 - val_accuracy: 0.7875 - val_loss: 0.4725
Epoch 6/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 221ms/step - accuracy: 0.9500 - loss: 0.2220 - val_accuracy: 0.8062 - val_loss: 0.4316
Epoch 7/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 222ms/step - accuracy: 0.9563 - loss: 0.1521 - val_accuracy: 0.8062 - val_loss: 0.4304
Epoch 8/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 181ms/step - accuracy: 0.9469 - loss: 0.1382 - val_accuracy: 0.

In [ ]:
model_name = 'bi directional - LSTM'

y_prob = BI_lstm_model.predict(X_test).ravel()
y_pred = (y_prob >= 0.5).astype(int)
print(f'\n{'='*50}')
print(f'  {model_name} Results')
print(f'{'='*50}')
print(classification_report(y_test, y_pred))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')

7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 96ms/step

  bi directional - LSTM Results
              precision    recall  f1-score   support

           0       0.70      0.73      0.71        96
           1       0.74      0.71      0.73       104

    accuracy                           0.72       200
   macro avg       0.72      0.72      0.72       200
weighted avg       0.72      0.72      0.72       200

Confusion Matrix:
[[70 26]
 [30 74]]
Accuracy: 0.7200


In [ ]:
deep_lstm_model = Sequential([

    Input(shape=(MAX_LEN,)),

    Embedding(input_dim=VOCAB_SIZE,
              output_dim=EMBED_DIM),

    LSTM(128, return_sequences=True),      #layer1

    LSTM(64),                              #layer2

    Dropout(0.3),

    Dense(1, activation='sigmoid')
])

deep_lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

deep_lstm_model.summary()

history_DEEP = deep_lstm_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 200, 64)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 200, 128)       │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 200, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 468,289 (1.79 MB)

 Trainable params: 468,289 (1.79 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 12s 407ms/step - accuracy: 0.4625 - loss: 0.6967 - val_accuracy: 0.4563 - val_loss: 0.7007
Epoch 2/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 11s 434ms/step - accuracy: 0.4969 - loss: 0.6930 - val_accuracy: 0.4563 - val_loss: 0.6938
Epoch 3/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 13s 673ms/step - accuracy: 0.5047 - loss: 0.6924 - val_accuracy: 0.4563 - val_loss: 0.7009
Epoch 4/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 20s 624ms/step - accuracy: 0.5094 - loss: 0.6938 - val_accuracy: 0.5437 - val_loss: 0.6928
Epoch 5/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 25s 856ms/step - accuracy: 0.5125 - loss: 0.6930 - val_accuracy: 0.4563 - val_loss: 0.6973
Epoch 6/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 19s 946ms/step - accuracy: 0.5094 - loss: 0.6942 - val_accuracy: 0.4563 - val_loss: 0.6955
Epoch 7/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 15s 664ms/step - accuracy: 0.5141 - loss: 0.6928 - val_accuracy: 0.4563 - val_loss: 0.6948
Epoch 8/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 15s 782ms/step - accuracy: 0.5188 - loss: 0.6928 - val_accu

In [ ]:
model_name = 'deep - LSTM'

y_prob = deep_lstm_model.predict(X_test).ravel()
y_pred = (y_prob >= 0.5).astype(int)
print(f'\n{'='*50}')
print(f'  {model_name} Results')
print(f'{'='*50}')
print(classification_report(y_test, y_pred))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')

7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 151ms/step

  deep - LSTM Results
              precision    recall  f1-score   support

           0       0.48      1.00      0.65        96
           1       0.00      0.00      0.00       104

    accuracy                           0.48       200
   macro avg       0.24      0.50      0.32       200
weighted avg       0.23      0.48      0.31       200

Confusion Matrix:
[[ 96   0]
 [104   0]]
Accuracy: 0.4800


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
def predict_sentiment(review_text, model):
    review = re.sub('[^a-zA-Z]', ' ', review_text)
    review = review.lower().split()
    review = [ps.stem(w) for w in review if w not in stops]
    review = ' '.join(review)

    seq    = tokenizer.texts_to_sequences([review])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')

    prob   = model.predict(padded, verbose=0)[0][0]


    label  = 'Positive 😊' if prob >= 0.5 else 'Negative 😞'
    return label, float(prob)



In [ ]:


test_reviews = [
    "Wow loved this place! The food was amazing and staff were so friendly.",
    "Horrible experience. The food was cold and service was terrible.",
    "Average food, nothing special but not bad either.",
    "Best burger I love ever had in my life! Will definitely come back.",
    "Waited 45 minutes for my order and it was completely wrong.",
    "Decent place. The pasta was okay but a bit overpriced.",
]

In [ ]:
models = [lstm_model , BI_lstm_model , deep_lstm_model]

In [ ]:


for mode in models:
  print(f"\n\n\n-------------Using {mode} model-----------")

  for review in test_reviews:
      label, confidence = predict_sentiment(review , mode)
      bar = '█' * int(confidence * 20) + '-' * (20 - int(confidence * 20))
      print(f"\n Review    : {review[:60]}...")
      print(f" Sentiment : {label}")
      print(f" Confidence: [{bar}] {confidence:.2%}")
      print("-"*55)




-------------Using <Sequential name=sequential, built=True> model-----------

 Review    : Wow loved this place! The food was amazing and staff were so...
 Sentiment : Negative 😞
 Confidence: [█████████-----------] 48.22%
-------------------------------------------------------

 Review    : Horrible experience. The food was cold and service was terri...
 Sentiment : Negative 😞
 Confidence: [█████████-----------] 48.22%
-------------------------------------------------------

 Review    : Average food, nothing special but not bad either....
 Sentiment : Negative 😞
 Confidence: [█████████-----------] 48.22%
-------------------------------------------------------

 Review    : Best burger I love ever had in my life! Will definitely come...
 Sentiment : Negative 😞
 Confidence: [█████████-----------] 48.22%
-------------------------------------------------------

 Review    : Waited 45 minutes for my order and it was completely wrong....
 Sentiment : Negative 😞
 Confidence: [█████████---